In [1]:
import model as m
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import Evaluate_results as er

2024-03-12 11:15:12.895188: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-03-12 11:15:12.896970: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-03-12 11:15:12.922680: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-12 11:15:12.922708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-12 11:15:12.923423: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [2]:
dataset_path = 'Results/'


In [3]:

selected_genes_GB = pd.read_csv('Results/selected_genes_GB_ACC.csv')
selected_genes_EN = pd.read_csv('Results/selected_genes_Ensembl.csv')
microarray_dataframe = pd.read_csv('Data/trainign_dataset_51270_microarray_adjusted.csv')
rnaseq_dataframe = pd.read_csv('Data/RNAseq_All_abundances.csv')

/tmp/ipykernel_398150/702159296.py:3: DtypeWarning: Columns (51280) have mixed types. Specify dtype option on import or set low_memory=False.
  microarray_dataframe = pd.read_csv('Data/trainign_dataset_51270_microarray_adjusted.csv')


In [4]:

columns_to_keep = (selected_genes_GB["Genes"]).to_list()
columns_to_keep.extend(['Experiment', 'Sex', 'Age', 'Status', "Sample"])

In [5]:
type(columns_to_keep)

list

In [6]:
microarray_dataframe.columns

Index(['Sample', 'U48705', 'M87338', 'X51757', 'X69699', 'L36861', 'L13852',
       'X55005', 'X79510', 'M21121',
       ...
       'AI571298', 'AA149545', 'C18318', 'AI219073', 'AI205180', 'AI363375',
       'Experiment', 'Sex', 'Age', 'Status'],
      dtype='object', length=51281)

In [7]:
# Keep only the columns in DataFrame B["genes"] and the specified columns
microarray_dataframe_f = microarray_dataframe[[col for col in columns_to_keep if col in microarray_dataframe.columns]]
microarray_dataframe_f


,NM_001091,NM_033128,BC010094,NM_139011,NM_004796,AB049740,BC036055,AK024889,BC001292,NM_018891,...,AF039555,NM_153038,NM_003468,BC039525,BC033791,Experiment,Sex,Age,Status,Sample
0,-22.573949,-599.120355,52.566712,-113.256419,-2.844541,206.793725,9.526148,85.097007,0.151063,25.562121,...,180.505833,242.366188,-149.135786,29.635569,151.393687,GSE13070,Male,52.50,IRd,GSM342608
1,40.582949,-305.994170,-90.104974,-104.810929,7.044176,125.447690,-46.529968,70.789344,95.232322,-17.322060,...,12.437130,196.003280,-86.422359,-85.657853,222.265852,GSE13070,Male,50.60,IRd,GSM342609
2,106.475469,242.852257,-216.716756,-165.336905,-13.047183,50.805966,-7.543569,11.688358,297.781805,-46.315497,...,-52.593207,-291.440966,-266.200843,66.517060,-290.744506,GSE13070,Male,56.08,IRd � TZD,GSM342610
3,-55.282322,435.322594,98.007517,-163.929326,37.338180,-117.026109,4.046982,43.296157,101.540488,-38.670926,...,-344.787770,138.492936,219.712059,-4.456571,365.310570,GSE13070,Male,54.84,IRd � TZD,GSM342611
4,-35.776237,561.142411,137.472522,-78.770693,-19.011805,103.993774,33.128726,92.391118,-121.883287,216.210269,...,351.099986,102.264004,-137.522188,20.337713,123.434976,GSE13070,Male,51.14,IRd,GSM342614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,-24.265458,-2868.647882,-91.611689,-5349.074372,35.441755,41.616739,-9.305701,17.866800,1.900009,-14.154503,...,40.946122,5.917648,36.035558,203.927666,0.269241,GSE9676,Female,65.10,NaN,GSM244612
773,-10.727633,-6264.922043,-310.337465,-7278.419523,0.417604,45.353801,-5.381675,17.866800,27.993254,-12.031699,...,161.697761,35.321227,-4.272834,-235.190606,7.171185,GSE9676,Female,68.40,NaN,GSM244613
774,-21.559355,320.624735,-57.493239,-6915.400963,0.877969,47.282140,-4.306533,17.866800,31.760710,-11.353313,...,0.138170,-43.589751,-59.249939,-126.909946,11.163149,GSE9676,Female,65.90,NaN,GSM244614
775,299.088280,-1737.069863,688.415455,-7244.754032,-5.085245,30.438991,-11.493155,17.866800,2.659549,-18.771733,...,33.699626,316.409398,222.389069,-439.584904,-12.883048,GSE9676,Female,67.30,NaN,GSM244615


In [8]:
microarray_dataframe_f.to_csv('Results/microarray_w_selected_genes.csv', index=False)

In [9]:
[col for col in rnaseq_dataframe.columns if any(prefix in col for prefix in columns_to_keep)]

['Sample', 'Experiment', 'Age', 'Sample.1']

In [10]:
rnaseq_dataframe.columns

Index(['Sample', 'ENSG00000000003.14', 'ENSG00000000005.5',
       'ENSG00000000419.12', 'ENSG00000000457.13', 'ENSG00000000460.16',
       'ENSG00000000938.12', 'ENSG00000000971.15', 'ENSG00000001036.13',
       'ENSG00000001084.11',
       ...
       'ENSG00000285472.1', 'ENSG00000285476.1', 'ENSG00000285480.1',
       'ENSG00000285491.1', 'ENSG00000285505.1', 'ENSG00000285508.1',
       'ENSG00000285509.1', 'Experiment', 'Age', 'Sample.1'],
      dtype='object', length=34509)

In [11]:
columns_to_keep = (selected_genes_EN["Genes"]).to_list()
columns_to_keep.extend(['Experiment', 'Age', "Sample"])
rnaseq_dataframe_f = rnaseq_dataframe[[col for col in rnaseq_dataframe.columns if any(col.startswith(prefix) for prefix in columns_to_keep)]]
rnaseq_dataframe_f.drop(columns=['Sample.1'], inplace=True)
rnaseq_dataframe_f.index = rnaseq_dataframe_f['Sample']
rnaseq_dataframe_f.drop(columns=['Sample'], inplace=True)
rnaseq_dataframe_f

/tmp/ipykernel_398150/4267592259.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rnaseq_dataframe_f.drop(columns=['Sample.1'], inplace=True)
/tmp/ipykernel_398150/4267592259.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rnaseq_dataframe_f.drop(columns=['Sample'], inplace=True)


,ENSG00000002726.20,ENSG00000006747.14,ENSG00000007541.16,ENSG00000010704.18,ENSG00000021645.18,ENSG00000033170.16,ENSG00000046889.18,ENSG00000053747.15,ENSG00000054611.13,ENSG00000058085.14,...,ENSG00000248383.4,ENSG00000254901.7,ENSG00000267534.3,ENSG00000270882.2,ENSG00000274618.1,ENSG00000275183.1,ENSG00000275379.1,ENSG00000276293.4,Experiment,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13388732,0.123684,5.189720,0.251315,1.069411,2.144491,10.444193,3.170066,3.671170,0.254691,0.510116,...,0.206455,4.672100,0.362761,0.279098,0.0,0.000000,0.000000,1.672340,GSE164471,23
SRR13388733,0.059570,1.616655,0.043431,0.403343,0.549351,12.189761,6.183019,3.440102,0.543939,0.333878,...,0.089256,2.225120,0.355544,1.083224,0.0,0.075328,0.000000,5.335630,GSE164471,28
SRR13388734,1.147721,5.342953,0.224035,1.673538,4.280046,27.579573,5.055825,3.793979,2.330694,0.995536,...,0.315819,12.796630,1.001110,0.000000,0.0,0.770372,0.000000,6.866489,GSE164471,31
SRR13388735,0.000000,0.875562,0.108086,0.281677,0.247601,3.559914,2.053541,1.075153,0.191196,0.072568,...,0.019302,0.918402,0.213198,1.284540,0.0,0.079948,0.000000,1.529065,GSE164471,31
SRR13388736,1.490298,17.709070,1.319079,2.946214,9.254425,70.202621,10.398111,10.031909,1.461193,2.738276,...,0.804696,16.095000,3.236510,1.634110,0.0,0.792041,0.898364,2.971810,GSE164471,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR12604223,0.044758,0.671229,5.466753,0.328066,0.038664,1.453121,0.011478,0.612787,3.104248,0.218998,...,0.016693,4.229891,0.345596,1.069648,0.0,0.126007,0.000000,17.364780,GSE157585,24
SRR12604224,0.111376,0.096092,7.524413,0.748171,0.021236,1.073830,0.038697,0.439051,2.250140,0.093634,...,0.000000,3.745153,0.160763,0.583354,0.0,0.085428,0.000000,19.851500,GSE157585,24
SRR12604225,0.096882,0.661991,9.661184,0.758285,0.101350,0.911192,0.028430,0.451949,3.638220,0.151526,...,0.000000,3.293234,0.209363,2.854234,0.0,0.053055,0.000000,17.494140,GSE157585,24


In [12]:
rnaseq_dataframe_f.to_csv('Results/rnaseq_w_selected_genes.csv', index=True)
